In [0]:
from typing import Any

In [0]:
import logging
from datetime import datetime
from py4j.java_gateway import JavaObject 
from pyspark.sql import SparkSession, DataFrame as SparkDataFrame
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    trim,
    max,
    to_date,
)

In [0]:
def load_extfiles_delta_table(
    spark: SparkSession,
    extfiles_catalog: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> SparkDataFrame:
    
    df = (
        spark.read.table(f"{extfiles_catalog}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

def load_extfiles_max_business_date_filter(
    spark: SparkSession,
    catalog_extfiles: str,
    table_name: str,
    ) -> str:

    max_business_date_dt = spark.read.table(f"{catalog_extfiles}.{table_name}").select(max("BusinessDate")).collect()[0][0]
    max_business_date_str = datetime.strftime(max_business_date_dt, "%Y-%m-%d")
    date_filter = f"businessdate = {max_business_date_str!r}"
    return date_filter

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')

catalog_extfiles = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_ext_lms_bucket_analysis_extfile(
    spark, curated_zone, catalog_extfiles, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_extfiles, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :

        # schema = (
        #     "ExtLMSBucketAnalysisId String, SourceSystem String, PortfolioKey String, PortfolioName String, "
        #     "IngestionTimestamp Timestamp, BatchId String, ExtLMSBucketAnalysisKey String, "
        #     "ExposureKey String, SourceSystemIdentifier String, Currency String, Bucket1 String, Bucket2 String, "
        #     "Bucket3 String, Bucket4 String, Bucket5 String, Bucket6 String, Bucket7 String, Bucket8 String, "
        #     "Bucket9 String, Bucket10 String, Bucket11 String, Bucket12 String, Bucket13 String, Bucket14 String, "
        #     "TotalDue String, Balance String, FileDate Date, BusinessDate Date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExtLmsBucketAnalysis"

        # ================== Create Delta Table ==================

        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================

        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  
                "extfile",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== extfile_lms_bucket_analysis_lms_bucket_analysis ==================

        date_filter = load_extfiles_max_business_date_filter(
            spark,
            catalog_extfiles,
            "extlms_bucket_analysis",
            )

        extfile_lms_bucket_analysis_extra_filters = None

        if extfile_lms_bucket_analysis_extra_filters is not None:
            extfile_lms_bucket_analysis_extra_filters.append(date_filter)
            extfile_lms_bucket_analysis_final_filters = " AND ".join(
                extfile_lms_bucket_analysis_extra_filters
            )
        else:
            extfile_lms_bucket_analysis_final_filters = date_filter

        extfile_lms_bucket_analysis_selected_fields = [
            "ExposureKey", 
            "BusinessDate",
            "SourceSystemIndicator", 
            "Currency",
            "Bucket1",
            "Bucket2",
            "Bucket3",
            "Bucket4",
            "Bucket5",
            "Bucket6",
            "Bucket7",
            "Bucket8",
            "Bucket9",
            "Bucket10",
            "Bucket11",
            "Bucket12",
            "Bucket13",
            "Bucket14",
            "TotalDue", 
            "Balance", 
            "RefDate", 
        ]

        extfile_lms_bucket_analysis = load_extfiles_delta_table(
            spark,
            catalog_extfiles,  
            "extlms_bucket_analysis", 
            extfile_lms_bucket_analysis_final_filters,
            extfile_lms_bucket_analysis_selected_fields,
        )

        extfile_lms_bucket_analysis = (
            extfile_lms_bucket_analysis
            .drop("BusinessDate").withColumn('BusinessDate', lit(run_date))
            .withColumn("TransformedRefDate", to_date("RefDate", "yyyyMMdd"))
        )

        target_table_Extfile_extfile_lms_bucket_analysisId_primary_key_fields = [
            extfile_lms_bucket_analysis.ExposureKey,
            lit("extfile"),
        ]

        target_table_Extfile = extfile_lms_bucket_analysis.withColumn(
            "ExtLMSBucketAnalysisId",
            sha2(
                concat_ws(
                    ";",
                    *target_table_Extfile_extfile_lms_bucket_analysisId_primary_key_fields,
                ),
                256,
            ),
        )

        # ================== Selection of Final Columns ==================

        result_df_Extfile = target_table_Extfile.select(
            target_table_Extfile.ExtLMSBucketAnalysisId.cast("String"),
            lit("extfile").alias("SourceSystem"),
            lit("-9999").alias("PortfolioKey"),
            lit("Core Retail").alias("PortfolioName"),
            from_utc_timestamp(current_timestamp(), "Europe/Athens").alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            target_table_Extfile.ExposureKey.cast("String").alias("ExtLMSBucketAnalysisKey"),
            target_table_Extfile.ExposureKey.cast("String"),
            target_table_Extfile.SourceSystemIndicator.cast("String").alias("SourceSystemIdentifier"),
            target_table_Extfile.Currency.cast("string"),
            target_table_Extfile.Bucket1.cast("String").alias("Bucket1"),
            target_table_Extfile.Bucket2.cast("String").alias("Bucket2"),
            target_table_Extfile.Bucket3.cast("String").alias("Bucket3"),
            target_table_Extfile.Bucket4.cast("String").alias("Bucket4"),
            target_table_Extfile.Bucket5.cast("String").alias("Bucket5"),
            target_table_Extfile.Bucket6.cast("String").alias("Bucket6"),
            target_table_Extfile.Bucket7.cast("String").alias("Bucket7"),
            target_table_Extfile.Bucket8.cast("String").alias("Bucket8"),
            target_table_Extfile.Bucket9.cast("String").alias("Bucket9"),
            target_table_Extfile.Bucket10.cast("String").alias("Bucket10"),
            target_table_Extfile.Bucket11.cast("String").alias("Bucket11"),
            target_table_Extfile.Bucket12.cast("String").alias("Bucket12"),
            target_table_Extfile.Bucket13.cast("String").alias("Bucket13"),
            target_table_Extfile.Bucket14.cast("String").alias("Bucket14"),
            target_table_Extfile.TotalDue.cast("String"),
            target_table_Extfile.Balance.cast("String"),
            target_table_Extfile.TransformedRefDate.cast("date").alias("FileDate"),
            target_table_Extfile.BusinessDate.cast("date"),
        )

        result_df_Extfile.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_extfiles,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_ext_lms_bucket_analysis_extfile(spark, curated_zone, catalog_extfiles, reference_zone, run_date, schema, env_prefix)